# Thực hành Transformers

Trong bài này, ta sẽ thực hành cài đặt Transformer

### 1. Cài đặt và import thư viện

In [8]:
!which python3

/opt/homebrew/bin/python3


In [9]:
!pip3 install spacy dill
!pip3 install torchtext
!pip3 install pandas

You should consider upgrading via the '/opt/homebrew/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip' command.
You should consider upgrading via the '/opt/homebrew/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip' command.
You should consider upgrading via the '/opt/homebrew/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip' command.


In [10]:
!python3 -m spacy download en && python3 -m spacy download fr

⚠ As of spaCy v3.0, shortcuts like 'en' are deprecated. Please use the
full pipeline package name 'en_core_web_sm' instead.
     |████████████████████████████████| 13.6 MB 851 kB/s eta 0:00:01
You should consider upgrading via the '/opt/homebrew/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip' command.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ As of spaCy v3.0, shortcuts like 'fr' are deprecated. Please use the
full pipeline package name 'fr_core_news_sm' instead.
     |████████████████████████████████| 17.1 MB 233 kB/s eta 0:00:01


You should consider upgrading via the '/opt/homebrew/opt/python@3.9/bin/python3.9 -m pip install --upgrade pip' command.
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')


In [11]:
import torch.nn as nn
import torch
import torchtext
import copy
import math
import torch.nn.functional as F

In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### 2. Cài đặt từng module của Transformer

In [13]:
class Embedder(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, dim)
    
    def forward(self, x):
        return self.embed(x)

**Position Embedding Class**:

In [ ]:
# Positional encoding
class PositionalEncoder(nn.Module):
    def __init__(self, dim, max_seq_len=300):
        super().__init__()
        self.dim = dim
        
        # create a constant 'pe' matrix with values dependant on 
        # pos and i
        pe = torch.zeros(max_seq_len, dim)

        ########################
        ##   YOUR CODE HERE   ##
        ########################
        for pos in range(max_seq_len):
            for i in range(0, dim, 2):
                pe[pos, i] = math.sin(pos / (10000 ** (i/dim)))
                if (i + 1 < dim):
                    pe[pos, i+1] = math.cos(pos / (10000 ** (i/dim)))
        
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        # make embeddings relatively larger
        x = x *math.sqrt(self.dim)
        # add constant to embedding
        seq_len = x.size(1)
        x = x + Variable(self.pe[:, :seq_len], requires_grad=False).to(device)
        return x

**Multi Head Attention**: We first start with implementing attention function

Attention of $q$

In [ ]:
def attention(q, k, v, d_k, mask=None, dropout=None):
    # q, k: N D
    
    scores = torch.matmul(q, k.transpose(-2, -1)) /  math.sqrt(d_k)
    if mask is not None:
        mask = mask.unsqueeze(1)
        scores = scores.masked_fill(mask == 0, -1e9)
    
    scores = F.softmax(scores, dim=-1)
    
    if dropout is not None:
        scores = dropout(scores)
        
    output = torch.matmul(scores, v)
    return output

In [ ]:
# Multi-headed attention
class MultiHeadAttention(nn.Module):
    def __init__(self, heads, dim, dropout=0.1):
        super().__init__()
        self.dim = dim
        self.dim_head = dim//heads
        self.h = heads
        self.q_linear = nn.Linear(dim, dim)
        self.k_linear = nn.Linear(dim, dim)
        self.v_linear = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(dim, dim)
    
    def forward(self, q, k, v, mask=None):
        bs = q.size(0)
        # perform linear operation and split into h heads
        # bs, sl, self.h, dim_head
        k = self.k_linear(k).view(bs, -1, self.h, self.dim_head)
        q = self.q_linear(q).view(bs, -1, self.h, self.dim_head)
        v = self.v_linear(v).view(bs, -1, self.h, self.dim_head)
        # transpose to get dimensions bs * h * sl * dim
        k = k.transpose(1, 2)
        q = q.transpose(1, 2)
        v = v.transpose(1, 2)
        # calculate attention using the function we will define next
        scores = attention(q, k, v, self.dim_head, mask, self.dropout)
        # concatenate heads and put through final linear layer
        concat = scores.transpose(1,2).contiguous().view(bs, -1, self.dim)
        output = self.out(concat)
        return output

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=2048, dropout = 0.1):
        super().__init__() 
        # We set d_ff as a default to 2048
        self.linear_1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_2 = nn.Linear(d_ff, d_model)
    def forward(self, x):
        x = self.dropout(F.relu(self.linear_1(x)))
        x = self.linear_2(x)
        return x

In [ ]:
class Norm(nn.Module):
    def __init__(self, d_model, eps = 1e-6):
        super().__init__()
    
        self.size = d_model
        # create two learnable parameters to calibrate normalisation
        self.alpha = nn.Parameter(torch.ones(self.size))
        self.bias = nn.Parameter(torch.zeros(self.size))
        self.eps = eps
    def forward(self, x):
        norm = self.alpha * (x - x.mean(dim=-1, keepdim=True)) \
        / (x.std(dim=-1, keepdim=True) + self.eps) + self.bias
        return norm

In [ ]:
# build an encoder layer with one multi-head attention layer and one 
# feed-forward layer
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout = 0.1):
        super().__init__()
        self.norm_1 = Norm(d_model)
        self.norm_2 = Norm(d_model)
        self.attn = MultiHeadAttention(heads, d_model)
        self.ff = FeedForward(d_model)
        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)
        
    def forward(self, x, mask):
        ########################
        ##   YOUR CODE HERE   ##
        ########################
        x1 = self.dropout_1(self.attn(x, x, x, mask))
        x = self.norm_1(x + x1)
        x1 = self.dropout_2(self.ff(x))
        x = self.norm_2(x + x1)
        return x
    

In [ ]:
# build a decoder layer with two multi-head attention layers and
# one feed-forward layer
class DecoderLayer(nn.Module):
    def __init__(self, d_model, heads, dropout=0.1):
        super().__init__()
        self.norm_1 = Norm(d_model)
        self.norm_2 = Norm(d_model)
        self.norm_3 = Norm(d_model)
        
        self.dropout_1 = nn.Dropout(dropout)
        self.dropout_2 = nn.Dropout(dropout)
        self.dropout_3 = nn.Dropout(dropout)
        
        self.attn_1 = MultiHeadAttention(heads, d_model)
        self.attn_2 = MultiHeadAttention(heads, d_model)
        self.ff = FeedForward(d_model).cuda()
    
    def forward(self, x, e_outputs, src_mask, trg_mask):
        ########################
        ##   YOUR CODE HERE   ##
        ########################
        x1 = self.dropout_1(self.attn_1(x, x, x, trg_mask))
        x = self.norm_1(x1 + x)
        x1 = self.dropout_2(self.attn_2(x, e_outputs, e_outputs, src_mask))
        x = self.norm_2(x1 + x)
        x = self.norm_3(x + self.dropout_3(self.ff(x)))
        return x
        
        

def get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

In [15]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, N, heads):
        super().__init__()
        self.N = N
        self.embed = Embedder(vocab_size, d_model)
        self.pe = PositionalEncoder(d_model)
        self.layers = get_clones(EncoderLayer(d_model, heads), N)
        self.norm = Norm(d_model)
        
    def forward(self, src, mask):
        ########################
        ##   YOUR CODE HERE   ##
        ########################
        x = self.embed(src)
        x = self.pe(x)
        for i in range(self.N):
            x = self.layers[i](x, mask)
        return self.norm(x)
    
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, N, heads):
        super().__init__()
        self.N = N
        self.embed = Embedder(vocab_size, d_model)
        self.pe = PositionalEncoder(d_model)
        self.layers = get_clones(DecoderLayer(d_model, heads), N)
        self.norm = Norm(d_model)
    def forward(self, trg, e_outputs, src_mask, trg_mask):
        ########################
        ##   YOUR CODE HERE   ##
        ########################
        x = self.embed(trg)
        x = self.pe(x)
        for i in range(self.N):
            x = self.layers[i](x, e_outputs, src_mask, trg_mask)
        return self.norm(x)

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, trg_vocab, d_model, N, heads):
        super().__init__()
        self.encoder = Encoder(src_vocab, d_model, N, heads)
        self.decoder = Decoder(trg_vocab, d_model, N, heads)
        self.out = nn.Linear(d_model, trg_vocab)
    def forward(self, src, trg, src_mask, trg_mask):
        e_outputs = self.encoder(src, src_mask)
        d_output = self.decoder(trg, e_outputs, src_mask, trg_mask)
        output = self.out(d_output)
        return output# we don't perform softmax on the output as this will be handled 
# automatically by our loss function

### 3. Chuẩn bị và tiền xử lý dữ liệu

In [16]:
import spacy
import re

class tokenize:
    def __init__(self, lang):
        self.nlp = spacy.load(lang)

    def tokenizer(self, sentence):
        sentence = re.sub(
        r"[\*"“”
\…\+\-\/\=\(\)'‘’:\[\]\|’\!;]", " ", str(sentence))
        sentence = re.sub(r"[ ]+", " ", sentence)
        sentence = re.sub(r"\!+", "!", sentence)
        sentence = re.sub(r"\,+", ",", sentence)
        sentence = re.sub(r"\?+", "?", sentence)
        sentence = sentence.lower()
        return [tok.text for tok in self.nlp.tokenizer(sentence) if tok.text != " "]


In [ ]:
import numpy as np
import torch
from torch.autograd import Variable

def nopeak_mask(size, opt):
    np_mask = np.triu(np.ones((1, size, size)), k=1).astype('uint8')
    np_mask = Variable(torch.from_numpy(np_mask) == 0)
    np_mask = np_mask.to(device)
    return np_mask

def create_masks(src, trg, opt):
    src_mask = (src != opt.src_pad).unsqueeze(-2)

    if trg is not None:
        trg_mask = (trg != opt.trg_pad).unsqueeze(-2).to(device)
        size = trg.size(1)
        np_mask = nopeak_mask(size, opt)
        trg_mask = trg_mask & np_mask
    else:
        trg_mask = None
    return src_mask, trg_mask

class SimpleIterator:
    def __init__(self, src_data, trg_data, batch_size, device, SRC, TRG):
        self.src_data = src_data
        self.trg_data = trg_data
        self.batch_size = batch_size
        self.device = device
        self.SRC = SRC
        self.TRG = TRG

    def __iter__(self):
        for i in range(0, min(len(self.src_data), len(self.trg_data)), self.batch_size):
            src_batch = self.src_data[i:i+self.batch_size]
            trg_batch = self.trg_data[i:i+self.batch_size]

            src_tensor = self.SRC.process(src_batch)
            trg_tensor = self.TRG.process(trg_batch)

            yield src_tensor.to(self.device), trg_tensor.to(self.device)

    def __len__(self):
        return min(len(self.src_data), len(self.trg_data)) // self.batch_size

def batch_size_fn(new, count, sofar):
    global max_src_in_batch, max_tgt_in_batch
    if count == 1:
        max_src_in_batch = 0
        max_tgt_in_batch = 0
    max_src_in_batch = max(max_src_in_batch, len(new.src))
    max_tgt_in_batch = max(max_tgt_in_batch, len(new.trg) + 2)
    src_elements = count * max_src_in_batch
    tgt_elements = count * max_tgt_in_batch
    return max(src_elements, tgt_elements)


In [ ]:
import pandas as pd
import os
import dill as pickle

class Vocab:
    def __init__(self):
        self.stoi = {}
        self.itos = []

    def build_vocab(self, sentences, tokenizer):
        word_count = {}
        for sent in sentences:
            for word in tokenizer(sent):
                word_count[word] = word_count.get(word, 0) + 1

        self.stoi['<pad>'] = 0
        self.stoi['<sos>'] = 1
        self.stoi['<eos>'] = 2
        self.stoi['<unk>'] = 3

        self.itos = ['<pad>', '<sos>', '<eos>', '<unk>']
        for word, count in sorted(word_count.items(), key=lambda x: -x[1]):
            if word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

    def __len__(self):
        return len(self.itos)

class Field:
    def __init__(self, lower=False, tokenizer=None, init_token=None, eos_token=None):
        self.lower = lower
        self.tokenizer = tokenizer
        self.init_token = init_token
        self.eos_token = eos_token
        self.vocab = None

    def build_vocab(self, data):
        self.vocab = Vocab()
        self.vocab.build_vocab(data, self.tokenizer)

    def process(self, batch):
        sequences = []
        for sent in batch:
            if self.lower:
                sent = sent.lower()
            tokens = []
            if self.init_token:
                tokens.append(self.init_token)
            tokens += self.tokenizer(sent)
            if self.eos_token:
                tokens.append(self.eos_token)
            sequences.append(tokens)

        max_len = max(len(seq) for seq in sequences)
        padded = []
        for seq in sequences:
            pad_token = self.vocab.stoi['<pad>']
            padded_seq = seq + [pad_token] * (max_len - len(seq))
            padded.append(padded_seq)

        return torch.tensor(padded, dtype=torch.long)

def read_data(opt):
    if opt.src_data is not None:
        try:
            with open(opt.src_data, 'r', encoding='utf-8') as f:
                opt.src_data = f.read().strip().split('\n')
        except:
            print("error: '" + opt.src_data + "' file not found")
            quit()

    if opt.trg_data is not None:
        try:
            with open(opt.trg_data, 'r', encoding='utf-8') as f:
                opt.trg_data = f.read().strip().split('\n')
        except:
            print("error: '" + opt.trg_data + "' file not found")
            quit()

def create_fields(opt):
    spacy_langs = ['en', 'fr', 'de', 'es', 'pt', 'it', 'nl']
    src_lang = opt.src_lang[0:2]
    trg_lang = opt.trg_lang[0:2]
    if src_lang not in spacy_langs:
        print('invalid src language: ' + opt.src_lang + ' supported languages: ' + str(spacy_langs))
    if trg_lang not in spacy_langs:
        print('invalid trg language: ' + opt.trg_lang + ' supported languages: ' + str(spacy_langs))

    print("loading spacy tokenizers...")

    t_src = tokenize(opt.src_lang)
    t_trg = tokenize(opt.trg_lang)
    TRG = Field(lower=True, tokenizer=t_trg.tokenizer.tokenizer, init_token='<sos>', eos_token='<eos>')
    SRC = Field(lower=True, tokenizer=t_src.tokenizer.tokenizer)

    return(SRC, TRG)

def create_dataset(opt, SRC, TRG):
    print("creating dataset and iterator... ")

    mask = []
    for i, (src, trg) in enumerate(zip(opt.src_data, opt.trg_data)):
        if src.strip() and trg.strip():
            if src.count(' ') < opt.max_strlen and trg.count(' ') < opt.max_strlen:
                mask.append(i)

    filtered_src = [opt.src_data[i] for i in mask]
    filtered_trg = [opt.trg_data[i] for i in mask]

    opt.src_data = filtered_src
    opt.trg_data = filtered_trg

    opt.train = SimpleIterator(opt.src_data, opt.trg_data, opt.batchsize, device, SRC, TRG)

    print(f"Dataset created with {len(opt.src_data)} samples")
    return opt.train

def get_len(train):
    count = 0
    for _ in train:
        count += 1
    return count


### 4. Cài đặt giải thuật tối ưu và huấn luyện mô hình

In [17]:
# Optimizer
class CosineWithRestarts(torch.optim.lr_scheduler._LRScheduler):
    """
    Cosine annealing with restarts.
    Parameters
    ----------
    optimizer : torch.optim.Optimizer
    T_max : int
        The maximum number of iterations within the first cycle.
    eta_min : float, optional (default: 0)
        The minimum learning rate.
    last_epoch : int, optional (default: -1)
        The index of the last epoch.
    """

    def __init__(self,
                 optimizer: torch.optim.Optimizer,
                 T_max: int,
                 eta_min: float = 0.,
                 last_epoch: int = -1,
                 factor: float = 1.) -> None:
        # pylint: disable=invalid-name
        self.T_max = T_max
        self.eta_min = eta_min
        self.factor = factor
        self._last_restart: int = 0
        self._cycle_counter: int = 0
        self._cycle_factor: float = 1.
        self._updated_cycle_len: int = T_max
        self._initialized: bool = False
        super(CosineWithRestarts, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        """Get updated learning rate."""
        # HACK: We need to check if this is the first time get_lr() was called, since
        # we want to start with step = 0, but _LRScheduler calls get_lr with
        # last_epoch + 1 when initialized.
        if not self._initialized:
            self._initialized = True
            return self.base_lrs

        step = self.last_epoch + 1
        self._cycle_counter = step - self._last_restart

        lrs = [
            (
                self.eta_min + ((lr - self.eta_min) / 2) *
                (
                    np.cos(
                        np.pi *
                        ((self._cycle_counter) % self._updated_cycle_len) /
                        self._updated_cycle_len
                    ) + 1
                )
            ) for lr in self.base_lrs
        ]

        if self._cycle_counter % self._updated_cycle_len == 0:
            # Adjust the cycle length.
            self._cycle_factor *= self.factor
            self._cycle_counter = 0
            self._updated_cycle_len = int(self._cycle_factor * self.T_max)
            self._last_restart = step

        return lrs


In [20]:
import urllib.request
import os

os.makedirs('data', exist_ok=True)

url_en = 'https://raw.githubusercontent.com/SamLynnEvans/Transformer/master/data/english.txt'
url_fr = 'https://raw.githubusercontent.com/SamLynnEvans/Transformer/master/data/french.txt'

print('Downloading english.txt...')
urllib.request.urlretrieve(url_en, 'data/english.txt')
print('Downloading french.txt...')
urllib.request.urlretrieve(url_fr, 'data/french.txt')
print('Done!')


mkdir: data: File exists
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4782k  100 4782k    0     0  1478k      0  0:00:03  0:00:03 --:--:-- 1481k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 5799k  100 5799k    0     0  1535k      0  0:00:03  0:00:03 --:--:-- 1538k


In [ ]:

def get_model(opt, src_vocab, trg_vocab):
    
    assert opt.d_model % opt.heads == 0
    assert opt.dropout < 1

    model = Transformer(src_vocab, trg_vocab, opt.d_model, opt.n_layers, opt.heads)
       
    if opt.load_weights is not None:
        print("loading pretrained weights...")
        model.load_state_dict(torch.load(f'{opt.load_weights}/model_weights'))
    else:
        for p in model.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p) 
    
    if opt.device == 0:
        model = model.cuda()
    
    return model

In [ ]:
""" BAI TAP VE NHA """

import time

class Opt:
    pass

def train_model(model, opt):
    ########################
    ##   YOUR CODE HERE   ##
    ########################

    criterion = nn.CrossEntropyLoss(ignore_index=opt.trg_pad)
    model.train()

    for epoch in range(opt.epochs):
        total_loss = 0
        start_time = time.time()
        batch_count = 0

        for i, batch in enumerate(opt.train):
            src, trg = batch
            src = src.to(device)
            trg = trg.to(device)

            src_mask, trg_mask = create_masks(src, trg, opt)

            output = model(src, trg, src_mask, trg_mask)

            output = output.view(-1, output.shape[-1])
            trg = trg.view(-1)

            loss = criterion(output, trg)

            opt.optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.optimizer.step()

            total_loss += loss.item()
            batch_count += 1

            if batch_count % opt.printevery == 0:
                avg_loss = total_loss / opt.printevery
                elapsed = time.time() - start_time
                print(f"Epoch {epoch+1} | Batch {batch_count} | "
                      f"Loss: {avg_loss:.4f} | Time: {elapsed:.2f}s")
                total_loss = 0
                start_time = time.time()

        print(f"\n=== Epoch {epoch+1} completed ===\n")

    print("Training completed!")

def main():
    opt = Opt()
    opt.src_data = "data/english.txt"
    opt.trg_data = "data/french.txt"
    opt.src_lang = "en_core_web_sm"
    opt.trg_lang = 'fr_core_news_sm'
    opt.epochs = 2
    opt.d_model=512
    opt.n_layers=6
    opt.heads=8
    opt.dropout=0.1
    opt.batchsize=1500
    opt.printevery=100
    opt.lr=0.0001
    opt.max_strlen=80
    opt.checkpoint = 0
    opt.no_cuda = False
    opt.load_weights = None

    read_data(opt)
    SRC, TRG = create_fields(opt)
    create_dataset(opt, SRC, TRG)
    model = get_model(opt, len(SRC.vocab), len(TRG.vocab)).to(device)

    opt.optimizer = torch.optim.Adam(model.parameters(), lr=opt.lr, betas=(0.9, 0.98), eps=1e-9)

    if opt.checkpoint > 0:
        print("model weights will be saved every %d minutes and at end of epoch to directory weights/"%(opt.checkpoint))

    train_model(model, opt)


if __name__ == "__main__":
    main()
